# Dual-Stream AI-Generated & Deepfake Image Detection
### Integrating Spatial Visual Cues & 2D FFT Spectral Analysis

**Project Overview:**
This notebook implements the complete training and evaluation pipeline for AI-generated image detection.
It directly incorporates the professor's recommendation to perform **Spectral / Frequency Analysis** alongside spatial deep learning:

1. **Spatial Stream:** Pretrained ResNet-50 capturing high-level semantics, textures, and blending boundaries.
2. **Spectral Stream:** 2D Fast Fourier Transform (2D FFT) log-magnitude spectrum analyzed by a dedicated frequency CNN to detect periodic upsampling artifacts from generative models (GANs, Diffusion models).
3. **Calibrated Fusion Head:** Feature concatenation and soft-voting probability output.

In [ ]:
# Step 1: Verify Hardware & GPU Acceleration
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Install and Import Required Libraries
!pip install -q kagglehub scikit-learn matplotlib tqdm

import os
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 3: Dataset Ingestion (CIFAKE / AI-Generated Benchmark)
CIFAKE contains 60,000 synthetic images (generated by Stable Diffusion) and 60,000 real photos (from CIFAR-10), perfectly balanced with independent, clean train and test splits (no data leakage).

In [ ]:
# Download CIFAKE benchmark via kagglehub or direct mirror
import kagglehub

# Download latest version of CIFAKE
try:
    path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
    print("Path to dataset files:", path)
    DATASET_PATH = Path(path)
except Exception as e:
    print("kagglehub error, using local/drive data path:", e)
    DATASET_PATH = Path("/content/data")

## Step 4: Spectral / Frequency Analysis Module (2D FFT)
Real camera images follow natural scene statistics where spectral power decays smoothly according to $P(f) \propto 1/f^2$.
Generative models (GANs, Diffusion decoders) introduce periodic grid artifacts due to transposed convolutions, resulting in distinct high-frequency spikes and anomalies in the 2D Fourier magnitude spectrum.

In [ ]:
def compute_fft_2d_torch(image_tensor):
    """
    Computes normalized 2D FFT Log-Magnitude Spectrum in PyTorch.
    Input: (B, C, H, W) RGB -> Output: (B, 1, H, W) in [0, 1]
    """
    weights = torch.tensor([0.2989, 0.5870, 0.1140], device=image_tensor.device).view(1, 3, 1, 1)
    gray = torch.sum(image_tensor * weights, dim=1, keepdim=True)
    
    # 2D Fast Fourier Transform & zero-frequency center shift
    fft2 = torch.fft.fft2(gray)
    fft_shift = torch.fft.fftshift(fft2, dim=(-2, -1))
    
    # Log magnitude spectrum
    log_mag = torch.log1p(torch.abs(fft_shift))
    
    # Batch-wise normalization to [0, 1]
    min_val = log_mag.amin(dim=(-2, -1), keepdim=True)
    max_val = log_mag.amax(dim=(-2, -1), keepdim=True)
    denom = torch.clamp(max_val - min_val, min=1e-8)
    return (log_mag - min_val) / denom

print("Spectral analysis function ready.")

In [ ]:
# Scientific Demonstration: Plotting Spatial vs Frequency Domain
# Demonstrates periodic upsampling artifacts in AI images to include in your project report!
def visualize_spectral_differences(real_sample, fake_sample):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    
    # Real Spatial & FFT
    axes[0, 0].imshow(real_sample)
    axes[0, 0].set_title("Real Photo (Spatial Domain)")
    axes[0, 0].axis("off")
    
    real_gray = np.dot(np.array(real_sample)[...,:3], [0.2989, 0.5870, 0.1140])
    real_fft = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(real_gray))))
    axes[0, 1].imshow(real_fft, cmap="inferno")
    axes[0, 1].set_title("Real Photo 2D FFT (Smooth 1/f^2 Decay)")
    axes[0, 1].axis("off")
    
    # Fake Spatial & FFT
    axes[1, 0].imshow(fake_sample)
    axes[1, 0].set_title("AI Generated (Spatial Domain)")
    axes[1, 0].axis("off")
    
    fake_gray = np.dot(np.array(fake_sample)[...,:3], [0.2989, 0.5870, 0.1140])
    fake_fft = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(fake_gray))))
    axes[1, 1].imshow(fake_fft, cmap="inferno")
    axes[1, 1].set_title("AI Generated 2D FFT (Periodic Spikes/Artifacts)")
    axes[1, 1].axis("off")
    
    plt.tight_layout()
    plt.show()

print("Spectral visualizer function defined.")

## Step 5: Dual-Stream Model Architecture

In [ ]:
class SpectralCNNEncoder(nn.Module):
    def __init__(self, out_features=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, out_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
    def forward(self, x):
        return self.fc(self.features(x))


class DualStreamDetector(nn.Module):
    def __init__(self, backbone="resnet50", pretrained=True, num_classes=2):
        super().__init__()
        # 1. Spatial Stream
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        spatial_dim = resnet.fc.in_features
        resnet.fc = nn.Identity()
        self.spatial_encoder = resnet
        
        # 2. Spectral Stream
        spectral_dim = 256
        self.spectral_encoder = SpectralCNNEncoder(out_features=spectral_dim)
        
        # Auxiliary heads for individual stream performance
        self.spatial_head = nn.Linear(spatial_dim, num_classes)
        self.spectral_head = nn.Linear(spectral_dim, num_classes)
        
        # 3. Fusion Head
        total_dim = spatial_dim + spectral_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x_rgb, return_breakdown=False):
        spatial_emb = self.spatial_encoder(x_rgb)
        spectral_input = compute_fft_2d_torch(x_rgb)
        spectral_emb = self.spectral_encoder(spectral_input)
        
        fused_emb = torch.cat([spatial_emb, spectral_emb], dim=1)
        fused_logits = self.fusion_head(fused_emb)
        
        if return_breakdown:
            return {
                "logits": fused_logits,
                "spatial_logits": self.spatial_head(spatial_emb),
                "spectral_logits": self.spectral_head(spectral_emb)
            }
        return fused_logits

print("DualStreamDetector model class compiled successfully.")

## Step 6: Create DataLoaders for Training and Testing

In [ ]:
# Locate train and test directories in the downloaded dataset
train_dir = None
test_dir = None

for p in Path(DATASET_PATH).rglob("*"):
    if p.is_dir() and p.name.lower() == "train":
        train_dir = p
    elif p.is_dir() and p.name.lower() == "test":
        test_dir = p

print("Found train dir:", train_dir)
print("Found test dir:", test_dir)

img_size = 224
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

print(f"Total Train Samples: {len(train_dataset)}")
print(f"Total Test Samples:  {len(test_dataset)}")
print(f"Class Mapping: {train_dataset.class_to_idx}")

# Set batch size (64 runs smoothly on T4 GPU)
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready.")

## Step 7: Multi-Task Training Loop & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if scaler and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images, return_breakdown=True)
                loss = (criterion(outputs["logits"], labels) + 
                        0.3 * criterion(outputs["spatial_logits"], labels) + 
                        0.3 * criterion(outputs["spectral_logits"], labels))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images, return_breakdown=True)
            loss = (criterion(outputs["logits"], labels) + 
                    0.3 * criterion(outputs["spatial_logits"], labels) + 
                    0.3 * criterion(outputs["spectral_logits"], labels))
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs["logits"], dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
    return running_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images, return_breakdown=True)
            
            loss = criterion(outputs["logits"], labels)
            running_loss += loss.item() * images.size(0)
            
            probs = torch.softmax(outputs["logits"], dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(outputs["logits"], dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)
            
    val_acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return running_loss / len(loader.dataset), val_acc, prec, rec, f1, auc, all_labels, all_preds, all_probs

## Step 8: Execute Training Loop (5 Epochs)

In [ ]:
# Initialize DualStreamDetector (ResNet-18 backbone for fast training and inference)
EPOCHS = 5
model = DualStreamDetector(backbone="resnet18", pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

best_acc = 0.0
SAVE_PATH = Path("/content/dual_stream_resnet18_best.pth")

print("--- Starting Training on T4 GPU ---")
for epoch in range(1, EPOCHS + 1):
    start_t = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
    scheduler.step()
    
    val_loss, val_acc, prec, rec, f1, auc, labels, preds, probs = evaluate(model, test_loader, criterion, device)
    elapsed = time.time() - start_t
    
    print(f"Epoch [{epoch}/{EPOCHS}] ({elapsed:.1f}s) | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Test Loss: {val_loss:.4f} Acc: {val_acc:.4f} AUC: {auc:.4f} F1: {f1:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_accuracy": val_acc,
            "val_auc": auc,
            "class_to_idx": train_dataset.class_to_idx
        }, SAVE_PATH)
        print(f"  * New best model saved to {SAVE_PATH} with Accuracy: {val_acc:.4f}")

print(f"\nTraining complete! Peak Test Accuracy: {best_acc * 100:.2f}%")

## Step 9: Plot Confusion Matrix & ROC Curve (For Project Report)

In [ ]:
# Load best checkpoint and plot confusion matrix
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
_, test_acc, prec, rec, f1, auc, labels, preds, probs = evaluate(model, test_loader, criterion, device)

cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "AI-Gen"])
disp.plot(cmap="Blues")
plt.title(f"VeriSight Dual-Stream Test Confusion Matrix (Acc: {test_acc*100:.1f}%)")
plt.savefig("/content/test_confusion_matrix.png", dpi=200)
plt.show()

## Step 10: Download Model Weights to Your PC

In [ ]:
# Download the trained model file directly to your computer!
from google.colab import files

print("Preparing to download dual_stream_resnet18_best.pth...")
files.download(str(SAVE_PATH))
print("Download started! Once downloaded, move this file to your deepfake-detection/saved_models/ folder.")